# Molecular Graph Learning Curve

This notebook measures how predictive performance on the MoleculeNet HIV dataset changes as the number of streamed training graphs increases, using `NSPPK` features with `radius=1`, `distance=4`, `connector=1` and a linear `SGDClassifier`.

The evaluation uses a simple contiguous split: the first 2000 loaded molecules are materialized, balanced, and used as the fixed test set; training then starts from the following molecules and updates `SGDClassifier.partial_fit(...)` batch by batch while the same held-out test set is scored at fixed checkpoints.


In [ ]:
from pathlib import Path
import gc
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import RDLogger
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

try:
    import psutil
except ImportError:
    psutil = None

REPO_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(candidate for candidate in REPO_CANDIDATES if (candidate / 'src').exists())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nsppk import NSPPK
from utils import plot_series_with_band_loess

DATASET_FILE = REPO_ROOT / 'data' / 'HIV.csv'
NBIT = 14
RADIUS = 1
DISTANCE = 4
CONNECTOR = 1
LIMIT = 8000
TEST_PREFIX_SIZE = 2000
BALANCE_TEST_SET = True
TRAIN_SIZE_VALUES = [250, 500, 1000, 2000, 4000]
N_REPEATS = 3
BATCH_SIZE = 256
WARMUP_SIZE = 256
PARALLEL = True
RANDOM_STATE = 42

RDLogger.DisableLog('rdApp.*')

def current_rss_mb():
    if psutil is not None:
        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    import resource
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if sys.platform == 'darwin':
        return rss / (1024 ** 2)
    return rss / 1024

MEMORY_LOG = []
MEMORY_PEAK_MB = 0.0

def record_memory(stage, emit=True):
    global MEMORY_PEAK_MB
    rss_mb = float(current_rss_mb())
    MEMORY_PEAK_MB = max(MEMORY_PEAK_MB, rss_mb)
    event = {'stage': stage, 'rss_mb': rss_mb, 'peak_rss_mb': MEMORY_PEAK_MB}
    MEMORY_LOG.append(event)
    if emit:
        print(f"[memory] {stage:<28} rss={rss_mb:8.2f} MB  peak={MEMORY_PEAK_MB:8.2f} MB")
    return event


## Setup

Build a fixed balanced test set from the first `TEST_PREFIX_SIZE` loaded molecules, warm up the vectorizer on the first training slice after that prefix, and track process RSS so we can confirm memory stays bounded while the training stream advances.


In [ ]:
loader = NSPPK(radius=RADIUS, distance=DISTANCE, connector=CONNECTOR, nbits=NBIT, parallel=PARALLEL)
record_memory('start')

test_graphs = loader.load_from(
    DATASET_FILE,
    'smiles',
    limit=TEST_PREFIX_SIZE,
    balance=BALANCE_TEST_SET,
    random_state=RANDOM_STATE,
    label_extractor=lambda graph: int(graph.graph['HIV_active']),
)
y_test = np.asarray([int(graph.graph['HIV_active']) for graph in test_graphs])
record_memory('test set loaded')

train_limit = max(LIMIT - TEST_PREFIX_SIZE, 0)
warmup_size = min(WARMUP_SIZE, train_limit)
warmup_graphs = loader.load_from(
    DATASET_FILE,
    'smiles',
    start_after_instance=TEST_PREFIX_SIZE,
    limit=warmup_size,
)
y_warmup = np.asarray([int(graph.graph['HIV_active']) for graph in warmup_graphs])
record_memory('warmup loaded')

vectorizer = NSPPK(
    radius=RADIUS,
    distance=DISTANCE,
    connector=CONNECTOR,
    nbits=NBIT,
    dense=False,
    parallel=PARALLEL,
)
vectorizer.fit(warmup_graphs)
record_memory('vectorizer fitted')
X_test = vectorizer.transform(test_graphs)
record_memory('test transformed')

classes = np.array([0, 1])
class_weight = None
if len(np.unique(y_warmup)) == len(classes):
    class_weight_values = compute_class_weight('balanced', classes=classes, y=y_warmup)
    class_weight = {cls: float(weight) for cls, weight in zip(classes, class_weight_values)}

train_size_values = [n for n in TRAIN_SIZE_VALUES if n <= train_limit]
if not train_size_values or train_size_values[-1] != train_limit:
    train_size_values.append(train_limit)

print('test graphs:', len(test_graphs))
print(f'test positive rate: {y_test.mean():.3f}')
print('train pool limit:', train_limit)
print('warmup graphs:', len(warmup_graphs))
print('class weight for partial_fit:', class_weight)
print('stream batch size:', BATCH_SIZE)
print('train sizes:', train_size_values)
print(f'initial peak RSS: {MEMORY_PEAK_MB:.2f} MB')


## Incremental Learning Curve

Train `SGDClassifier` with `partial_fit(...)` on successive training batches loaded from after the test prefix. Metrics and memory usage are recorded whenever the cumulative streamed count reaches one of the requested train sizes.


In [ ]:
results = []

print(f"{'train':>8} | {'repeat':>6} | {'roc_auc':>8} | {'avg_prec':>8} | {'seconds':>8} | {'rss_mb':>8} | {'peak_mb':>8}")
print('-' * 79)

repeat_histories = {
    train_size: {'roc_auc': [], 'avg_precision': [], 'runtime_sec': [], 'rss_mb': [], 'peak_rss_mb': []}
    for train_size in train_size_values
}
stop_positions = sorted(set(range(BATCH_SIZE, train_limit, BATCH_SIZE)) | set(train_size_values) | {train_limit})

for repeat_idx in range(1, N_REPEATS + 1):
    classifier = SGDClassifier(
        loss='log_loss',
        alpha=1e-5,
        penalty='l2',
        random_state=RANDOM_STATE + repeat_idx,
        class_weight=class_weight,
    )

    t0 = time.perf_counter()
    start = 0
    for stop in stop_positions:
        batch_graphs = loader.load_from(
            DATASET_FILE,
            'smiles',
            start_after_instance=TEST_PREFIX_SIZE + start,
            limit=stop - start,
        )
        y_batch = np.asarray([int(graph.graph['HIV_active']) for graph in batch_graphs])
        X_batch = vectorizer.transform(batch_graphs)

        if start == 0:
            classifier.partial_fit(X_batch, y_batch, classes=classes)
        else:
            classifier.partial_fit(X_batch, y_batch)

        del batch_graphs, y_batch, X_batch
        gc.collect()

        if stop in repeat_histories:
            memory_event = record_memory(f'repeat {repeat_idx} train {stop}', emit=False)
            y_score = classifier.decision_function(X_test)
            runtime_sec = time.perf_counter() - t0
            roc_auc = roc_auc_score(y_test, y_score)
            avg_precision = average_precision_score(y_test, y_score)

            repeat_histories[stop]['roc_auc'].append(float(roc_auc))
            repeat_histories[stop]['avg_precision'].append(float(avg_precision))
            repeat_histories[stop]['runtime_sec'].append(float(runtime_sec))
            repeat_histories[stop]['rss_mb'].append(float(memory_event['rss_mb']))
            repeat_histories[stop]['peak_rss_mb'].append(float(memory_event['peak_rss_mb']))

            print(
                f"{stop:>8d} | {repeat_idx:>6d} | {roc_auc:>8.4f} | {avg_precision:>8.4f} | "
                f"{runtime_sec:>8.2f} | {memory_event['rss_mb']:>8.2f} | {memory_event['peak_rss_mb']:>8.2f}"
            )

        start = stop

for train_size in train_size_values:
    history = repeat_histories[train_size]
    results.append(
        {
            'train_size': train_size,
            'mean_roc_auc': float(np.mean(history['roc_auc'])),
            'std_roc_auc': float(np.std(history['roc_auc'], ddof=0)),
            'mean_avg_precision': float(np.mean(history['avg_precision'])),
            'std_avg_precision': float(np.std(history['avg_precision'], ddof=0)),
            'mean_runtime_sec': float(np.mean(history['runtime_sec'])),
            'std_runtime_sec': float(np.std(history['runtime_sec'], ddof=0)),
            'mean_rss_mb': float(np.mean(history['rss_mb'])),
            'mean_peak_rss_mb': float(np.mean(history['peak_rss_mb'])),
            'nbits': NBIT,
            'radius': RADIUS,
            'distance': DISTANCE,
            'connector': CONNECTOR,
        }
    )

results_df = pd.DataFrame(results)
print(f'final peak RSS: {MEMORY_PEAK_MB:.2f} MB')
results_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
frac = 0.8  # Local quadratic LOESS span; adjust between 0 and 1 for more or less smoothing.
x = results_df['train_size'].to_numpy()

plot_series_with_band_loess(
    axes[0],
    x,
    results_df['mean_roc_auc'],
    y_std=results_df['std_roc_auc'],
    frac=frac,
    label='Mean ROC-AUC',
)
axes[0].set_xlabel('Training graphs')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Learning curve: ROC-AUC')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

plot_series_with_band_loess(
    axes[1],
    x,
    results_df['mean_avg_precision'],
    y_std=results_df['std_avg_precision'],
    frac=frac,
    color='tab:green',
    label='Mean average precision',
)
axes[1].set_xlabel('Training graphs')
axes[1].set_ylabel('Average precision')
axes[1].set_title('Learning curve: average precision')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plot_series_with_band_loess(
    axes[2],
    x,
    results_df['mean_runtime_sec'],
    y_std=results_df['std_runtime_sec'],
    frac=frac,
    color='tab:red',
    label='Mean runtime',
)
axes[2].set_xlabel('Training graphs')
axes[2].set_ylabel('Seconds')
axes[2].set_title('End-to-end runtime')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()
